# dunnhumby Breakfast at the Frat: how much of each manufacturer's cereal volume sells on a price cut?

A category manager's first question about promotions: which suppliers' volume depends on deals, and how deep are those deals? Reads the canonical tables written by `make onboard CLIENT=client_a`. Units are store x product x week units over the whole analysis window.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

client = "client_a"
sales = pl.read_parquet(f"../data/canonical/{client}/sales.parquet")
products = pl.read_parquet(f"../data/canonical/{client}/products.parquet")
sales.select(
    first_week=pl.col("week_start").min(),
    last_week=pl.col("week_start").max(),
    stores=pl.col("store_id").n_unique(),
    rows=pl.len(),
)

In [ ]:
cut = pl.col("on_price_cut")
by_manufacturer = (
    sales.join(products, on="product_id")
    .group_by("manufacturer")
    .agg(
        products=pl.col("product_id").n_unique(),
        units=pl.col("units").sum(),
        promo_units=pl.col("units").filter(cut).sum(),
        weeks_on_cut=cut.mean(),
        depth_on_cut=pl.col("discount_depth").filter(cut).mean(),
    )
    .with_columns(promo_share=pl.col("promo_units") / pl.col("units"))
    .sort("units", descending=True)
)
by_manufacturer

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
data = by_manufacturer.sort("promo_share")
ax.barh(data["manufacturer"], data["promo_share"] * 100, color="#2f5d8a")
ax.set_xlabel("Share of units sold in price-cut weeks (%)")
ax.set_title("Promo dependence by manufacturer")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Answer:** run the notebook once the raw files are in `data/raw/client_a/`, then write the answer here.